# Chapter 05-10 · Decision trees and random forests

**Label:** Core  |  **Time:** ~60 minutes  |  **Difficulty:** the algorithm is simple enough to do by
hand; its strengths and its one hard limit are both surprising

**Prerequisites:** 05-07 for capacity and interactions, 05-08 for bias and variance, 05-04 for MSE.

**Position in the learning path:** module 05, chapter 10 of 12.

---

## Why this matters

Every model so far has been **a weighted sum of the columns**. That shape has consequences you have been
paying for all module: a curve had to be supplied by hand in 05-05, an interaction by hand in 05-07, and
every column had to be scaled before anything would train.

**A tree is not a weighted sum.** It asks a question about one column, splits the rows in two, and
repeats. That single change removes all three of those chores - trees bend, they find interactions
unprompted, and they do not care what units anything is in - and introduces one limitation so absolute
that it is worth knowing before you learn anything else about them.

**And 05-08 left a promise.** Averaging many fits destroys variance and leaves bias alone; a single tree
is a very low-bias, very high-variance model. That is the exact shape averaging was made for, and the
algorithm that does it is the random forest.

## What you will be able to do

- Find the best split in a set of rows by hand, and say what "best" means
- Explain why a tree needs no scaling and finds interactions without being told
- Use depth as a capacity dial and read the resulting curve
- Say what bagging does to bias and to variance, with measured numbers
- Recognise the one thing trees categorically cannot do
- Read a feature-importance table without being misled by it

## Warm-up: retrieve, do not reread

1. In 05-07, what did the main-effects model report for the promotion, and what was the truth?
2. In 05-08, what did averaging 25 fits do to bias and to variance?
3. In 05-09, why does a penalised model need its columns scaled?

<br>

*Answers: (1) +84.87, against a truth of 15 on weekdays and 135 at weekends. (2) variance fell by a factor
of 22, bias did not move at all. (3) the penalty is charged on the coefficient, which depends on the
units.*

## One split, by hand

A regression tree splits a set of rows into two groups, and predicts the **mean of the target** in each.
"Best" means the split that leaves the least squared error inside the groups:

$$\text{SSE}(\text{split}) = \sum_{\text{left}}(y_i - \bar{y}_{\text{left}})^2 + \sum_{\text{right}}(y_i - \bar{y}_{\text{right}})^2$$

There are only finitely many splits to try - one between each pair of adjacent values - so the algorithm
is: **try them all, keep the best.** Here are eight rows.

### Predict before running

Look at the numbers below and pick the threshold you would use. Then check the table.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree

# SYNTHETIC: eight rows, deliberately small enough to check on paper
small_x = np.array([1.0, 2.0, 3.0, 4.0, 6.0, 7.0, 8.0, 9.0])
small_y = np.array([3.0, 4.0, 3.0, 5.0, 12.0, 11.0, 13.0, 12.0])

total_sse = float(((small_y - small_y.mean()) ** 2).sum())
print("x:", small_x.tolist())
print("y:", small_y.tolist())
print("\nSSE about the overall mean of %.2f: %.3f\n" % (small_y.mean(), total_sse))

rows = []
for index in range(len(small_x) - 1):
    threshold = (small_x[index] + small_x[index + 1]) / 2
    left, right = small_y[small_x < threshold], small_y[small_x >= threshold]
    sse_left = float(((left - left.mean()) ** 2).sum())
    sse_right = float(((right - right.mean()) ** 2).sum())
    rows.append({"threshold": threshold, "left mean": left.mean(),
                 "right mean": right.mean(), "SSE left": sse_left,
                 "SSE right": sse_right, "total SSE": sse_left + sse_right,
                 "reduction": total_sse - sse_left - sse_right})
splits = pd.DataFrame(rows)
print(splits.to_string(index=False, float_format=lambda v: "%.3f" % v))

winner = splits.loc[splits["total SSE"].idxmin()]
print("\nbest threshold %.1f, leaving SSE %.3f of the original %.3f"
      % (winner["threshold"], winner["total SSE"], total_sse))

stump = DecisionTreeRegressor(max_depth=1).fit(small_x.reshape(-1, 1), small_y)
print("sklearn's depth-1 tree splits at %.1f - the same answer"
      % stump.tree_.threshold[0])

**The split at 5.0 leaves an SSE of 4.750 out of 140.875 - it removes 97% of the squared error with one
question.**

Three things in that table are worth pausing on.

**The prediction is a mean, so the "model" is two numbers.** Left of 5.0 predict 3.75, right of 5.0
predict 12.0. That is the entire depth-1 tree.

**Nothing about the units mattered.** The algorithm only ever asked "is x less than this?", and the answer
is unchanged if you measure x in metres, kilometres or furlongs. **A tree is invariant to any monotone
transform of any feature** - which is why scaling, the non-negotiable of 05-06 and 05-09, is simply not a
consideration here.

**And the search is exhaustive, not clever.** Seven candidate thresholds, seven evaluations, take the
minimum. Repeat inside each group and you have a tree. That is the whole algorithm.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12.8, 4.4))

left.plot(splits["threshold"], splits["total SSE"], "o-", color="#0072B2", linewidth=2.4,
          markersize=9)
best_index = int(splits["total SSE"].idxmin())
left.plot([splits.loc[best_index, "threshold"]], [splits.loc[best_index, "total SSE"]],
          "*", color="#D55E00", markersize=22, label="the winner, 5.0")
left.axhline(total_sse, color="#000000", linestyle="--", linewidth=1.6,
             label="no split at all, %.1f" % total_sse)
left.set_xlabel("candidate threshold")
left.set_ylabel("SSE left inside the two groups")
left.set_title("Every split, scored", fontsize=11)
left.legend(fontsize=9)

right.scatter(small_x, small_y, s=90, color="#666666", zorder=3, label="the eight rows")
threshold = splits.loc[best_index, "threshold"]
right.axvline(threshold, color="#D55E00", linewidth=2.2, linestyle="--",
              label="split at %.1f" % threshold)
right.hlines(splits.loc[best_index, "left mean"], 0.5, threshold, color="#009E73",
             linewidth=3.5, label="predict %.2f" % splits.loc[best_index, "left mean"])
right.hlines(splits.loc[best_index, "right mean"], threshold, 9.5, color="#0072B2",
             linewidth=3.5, label="predict %.2f" % splits.loc[best_index, "right mean"])
right.set_xlabel("x")
right.set_ylabel("y")
right.set_title("The whole depth-1 model: two numbers", fontsize=11)
right.legend(fontsize=8.5)

plt.tight_layout()
plt.show()

## The interaction, found without being asked

05-07 spent a section arguing that a linear model needs `promo x weekend` supplied by hand, and that
without it the model reports a promotion effect of +84.87 which is correct for nobody.

**A tree is never told about the interaction and does not need to be.** Splitting on `weekend` and then
splitting on `promo` *inside each branch* is an interaction - the effect of the second question depends
on the answer to the first, which is the definition.

In [ ]:
# SYNTHETIC: 05-07's 600 shop-days.
# TRUTH: 200 + 0.9 footfall + 15 promo + 40 weekend + 120 (promo AND weekend), noise sd 25
shop_rng = np.random.default_rng(9)
n_days = 600
footfall = shop_rng.uniform(50, 400, n_days)
promo = shop_rng.integers(0, 2, n_days).astype(float)
weekend = shop_rng.integers(0, 2, n_days).astype(float)
sales = (200 + 0.9 * footfall + 15 * promo + 40 * weekend
         + 120 * promo * weekend + shop_rng.normal(0, 25, n_days))
shop = pd.DataFrame({"footfall": footfall, "promo": promo, "weekend": weekend})

train_shop, test_shop, train_sales, test_sales = train_test_split(
    shop, sales, test_size=0.3, random_state=0)


def score(model, X=test_shop, y=test_sales):
    prediction = model.predict(X)
    return float(np.sqrt(((y - prediction) ** 2).mean()))


results = [
    {"model": "linear, main effects only", "given the interaction?": "no",
     "test RMSE": score(LinearRegression().fit(train_shop, train_sales))},
]
with_product = train_shop.assign(px=train_shop.promo * train_shop.weekend)
test_product = test_shop.assign(px=test_shop.promo * test_shop.weekend)
results.append({"model": "linear, with promo x weekend", "given the interaction?": "YES",
                "test RMSE": score(LinearRegression().fit(with_product, train_sales),
                                   test_product)})
for depth in [2, 3, 5, None]:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(train_shop, train_sales)
    results.append({"model": "tree, max_depth %s" % depth, "given the interaction?": "no",
                    "test RMSE": score(tree)})
forest = RandomForestRegressor(n_estimators=300, random_state=0).fit(train_shop, train_sales)
results.append({"model": "random forest of 300", "given the interaction?": "no",
                "test RMSE": score(forest)})

print(pd.DataFrame(results).to_string(index=False, float_format=lambda v: "%.3f" % v))
print("\nthe noise this data was built with has sd 25")

**The tree at depth 5, given no interaction column, reaches RMSE 27.94 against the linear main-effects
model's 40.65.** It found the structure by itself.

**And it does not beat the linear model that *was* given the interaction - 23.41.** That gap is the
honest half of the story, and it has a specific cause: `footfall` enters the truth as a straight line,
and **a tree has to approximate a straight line with a staircase.** Every split it spends on approximating
that slope is a split it does not spend elsewhere.

> **A tree gets non-linearity and interactions for free, and pays for smooth trends.** A linear model is
> the exact opposite. Neither is better; they are wrong in different places.

**The unrestricted tree is worse than the depth-5 one** - 36.22 against 27.94 - which is 05-07's
overfitting arriving in a new form. Depth is this model's capacity dial, and the next section turns it.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.4))
shallow = DecisionTreeRegressor(max_depth=3, random_state=0).fit(train_shop, train_sales)
plot_tree(shallow, feature_names=list(shop.columns), filled=True, rounded=True,
          fontsize=8, precision=1, ax=ax, impurity=False)
ax.set_title("A depth-3 tree on the shop data. Read the top two levels.", fontsize=12)
plt.tight_layout()
plt.show()

**Read down the right-hand side.** The root splits on `footfall <= 246.0`, then **both** children split
on `weekend <= 0.5`, and inside the weekend branches the next question is `promo <= 0.5`.

**That nesting is the interaction.** The tree asks about the promotion *only after* it knows whether it is
the weekend, so the answer it gives is allowed to differ between the two - which is precisely what a
single `promo` coefficient could not do. Compare the two leaves under the right-most branch: **526.6 with
no promotion and 668.4 with one**, a gap of 142 on high-footfall weekends, against a much smaller gap in
the weekday branches.

**And notice the root is `footfall`, not a flag.** The tree spent its first and most valuable split
approximating a straight line, which is the staircase problem in one picture - and the reason it does not
catch the linear model that was handed the interaction.

**And notice what the leaves contain: `value`, which is a mean.** There are no coefficients anywhere in
this model. That is why nothing needed scaling, and it is also why there is nothing to report as an
effect size - a point the feature-importance section returns to.

## Depth is the capacity dial

In [ ]:
# SYNTHETIC: 400 points on a smooth curve. TRUTH: 3 sin(1.2x) + 0.5x, noise sd 1.5
def true_curve(x):
    return np.sin(1.2 * x) * 3 + 0.5 * x


NOISE_SD = 1.5
curve_rng = np.random.default_rng(7)
curve_x = curve_rng.uniform(-4, 4, 400)
curve_y = true_curve(curve_x) + curve_rng.normal(0, NOISE_SD, 400)
fit_x, held_x, fit_y, held_y = train_test_split(curve_x, curve_y, test_size=0.5,
                                                random_state=0)

depth_rows = []
for depth in [1, 2, 3, 4, 6, 8, 12, None]:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(
        fit_x.reshape(-1, 1), fit_y)
    depth_rows.append({
        "max_depth": str(depth), "leaves": tree.get_n_leaves(),
        "train RMSE": float(np.sqrt(((fit_y - tree.predict(fit_x.reshape(-1, 1))) ** 2).mean())),
        "held-out RMSE": float(np.sqrt(((held_y - tree.predict(held_x.reshape(-1, 1))) ** 2).mean()))})
print("%d rows to fit, noise floor %.2f\n" % (len(fit_x), NOISE_SD))
print(pd.DataFrame(depth_rows).to_string(index=False, float_format=lambda v: "%.4f" % v))

> **At `max_depth=None` the tree has 200 leaves for 200 rows and a training RMSE of exactly 0.0000.**

That is not an approximation to zero. **Every row has its own leaf**, so the tree predicts each training
row's target perfectly by storing it. It is a lookup table with a decision procedure attached, and its
held-out RMSE is 2.0709 - worse than the depth-1 stump's 2.1852 by only a hair, from a model that fits
the training data perfectly.

**A tree left unrestricted will always memorise.** Unlike a polynomial, which needs enough degrees, a tree
can keep splitting until every leaf is pure, and by default it does. **Every tree you fit needs a stopping
rule** - `max_depth`, `min_samples_leaf`, `min_samples_split`, or `ccp_alpha` - and the defaults do not
provide one.

The best held-out RMSE here is **1.5867 at depth 3**, with 8 leaves, against a noise floor of 1.50.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3), sharey=True)
grid = np.linspace(-4, 4, 500)

for ax, depth in zip(axes, [1, 3, None]):
    tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(
        fit_x.reshape(-1, 1), fit_y)
    ax.scatter(fit_x, fit_y, s=16, alpha=0.45, color="#999999", label="the 200 fitting rows")
    ax.plot(grid, true_curve(grid), color="#000000", linewidth=2.4, linestyle="--",
            label="the truth")
    ax.plot(grid, tree.predict(grid.reshape(-1, 1)), color="#D55E00", linewidth=2.2,
            label="the tree")
    ax.set_xlabel("x")
    ax.set_title("max_depth %s: %d leaves" % (depth, tree.get_n_leaves()), fontsize=11)
    ax.legend(fontsize=8, loc="upper left")

axes[0].set_ylabel("y")
plt.tight_layout()
plt.show()

**Every tree prediction is a horizontal step**, because every leaf predicts one number. That is the shape
of the model class, and all three panels show it:

- **Depth 1**: two steps. Too stiff to be anything.
- **Depth 3**: eight steps, tracking the curve about as well as eight numbers can.
- **Unrestricted**: 200 steps, one per row, passing through every point including the noise.

**This is why a tree cannot represent a straight line efficiently** - the staircase in the shop data,
where the linear model with an interaction won. It is also why the right panel's steps are wider where
data is sparse and narrower where it is dense: the tree spends capacity where the rows are.

## Bagging: 05-08's finding becomes an algorithm

05-08 established that **averaging many fits destroys variance and leaves bias untouched**, and measured
it: 25 averaged polynomial fits cut variance by a factor of 22. The caveat was that those 25 fits used
independent samples, which nobody has.

**Bootstrap aggregating - bagging - approximates independent samples by resampling the one dataset with
replacement.** A random forest is bagging applied to trees, plus one extra idea. Here is the measurement.

In [ ]:
EVALUATION_GRID = np.linspace(-4, 4, 60)


def bias_and_variance(make_model, rows=60, datasets=120, seed=0):
    rng = np.random.default_rng(seed)
    predictions = np.zeros((datasets, len(EVALUATION_GRID)))
    for run in range(datasets):
        x = rng.uniform(-4, 4, rows)
        y = true_curve(x) + rng.normal(0, NOISE_SD, rows)
        predictions[run] = make_model().fit(x.reshape(-1, 1),
                                            y).predict(EVALUATION_GRID.reshape(-1, 1))
    bias_squared = float(((predictions.mean(axis=0) - true_curve(EVALUATION_GRID)) ** 2).mean())
    variance = float(predictions.var(axis=0).mean())
    return {"bias squared": bias_squared, "variance": variance,
            "noise": NOISE_SD ** 2,
            "total": bias_squared + variance + NOISE_SD ** 2}


decomposition = []
for label, make in [
        ("one tree, unrestricted", lambda: DecisionTreeRegressor(random_state=0)),
        ("one tree, max_depth 3", lambda: DecisionTreeRegressor(max_depth=3, random_state=0)),
        ("forest of 10 trees", lambda: RandomForestRegressor(n_estimators=10, random_state=0)),
        ("forest of 100 trees", lambda: RandomForestRegressor(n_estimators=100, random_state=0))]:
    row = bias_and_variance(make)
    row["model"] = label
    decomposition.append(row)
print(pd.DataFrame(decomposition)[["model", "bias squared", "variance", "noise", "total"]]
      .to_string(index=False, float_format=lambda v: "%.4f" % v))

**A single unrestricted tree: bias² 0.0189, variance 2.2780. A forest of 100: bias² 0.0097, variance
1.0624.**

> **Bias stayed near zero and variance halved.** Exactly the effect 05-08 predicted, from the same
> mechanism, with the trees supplying the low-bias high-variance model that averaging was made for.

**And the factor is 2.14, not 100.** 05-08's caveat is now measured: bootstrap resamples share about
two-thirds of their rows with each other, so the trees are correlated and their errors do not cancel the
way independent ones would. **Bagging gets the direction of the theory and a fraction of its magnitude.**

**The second idea in a random forest exists to recover some of that fraction.** At each split, the tree is
allowed to consider only a random subset of the features (`max_features`). That makes the trees *less*
individually accurate and *less like each other* - and since averaging removes the disagreement but not
the shared error, trading accuracy for decorrelation is usually worth it. With one feature, as here,
there is nothing to subset, which is part of why the forest only halves the variance.

**Note also that the forest of 100 barely beats a depth-3 tree** - total 3.3221 against 3.3963. On a
one-dimensional smooth curve, restricting depth is nearly as good and far cheaper. **Forests earn their
keep when there are many features and interactions to find**, which is exactly where the depth-3 tree
would be helpless.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3), sharey=True)

for ax, (label, make) in zip(axes, [
        ("one tree, unrestricted", lambda: DecisionTreeRegressor(random_state=0)),
        ("forest of 10", lambda: RandomForestRegressor(n_estimators=10, random_state=0)),
        ("forest of 100", lambda: RandomForestRegressor(n_estimators=100, random_state=0))]):
    rng = np.random.default_rng(0)
    collected = []
    for _ in range(25):
        x = rng.uniform(-4, 4, 60)
        y = true_curve(x) + rng.normal(0, NOISE_SD, 60)
        collected.append(make().fit(x.reshape(-1, 1),
                                    y).predict(EVALUATION_GRID.reshape(-1, 1)))
    collected = np.array(collected)
    for single in collected:
        ax.plot(EVALUATION_GRID, single, color="#0072B2", alpha=0.2, linewidth=1.1)
    ax.plot(EVALUATION_GRID, collected.mean(axis=0), color="#D55E00", linewidth=3,
            label="the average fit")
    ax.plot(EVALUATION_GRID, true_curve(EVALUATION_GRID), color="#000000", linewidth=2.4,
            linestyle="--", label="the truth")
    ax.set_xlabel("x")
    ax.set_title("%s\nvariance %.2f" % (label, collected.var(axis=0).mean()), fontsize=11)
    ax.legend(fontsize=8, loc="upper left")

axes[0].set_ylabel("prediction")
plt.tight_layout()
plt.show()

**This is 05-08's spray figure with trees in it, and the narrowing is the algorithm working.**

The single trees scatter wildly and the orange average sits on the truth - a textbook low-bias,
high-variance model. Move right and the blue spray tightens while the orange line barely moves.

**The steps also get smaller.** Each tree in the forest produces a staircase, but they step in different
places, so the average of many staircases is nearly smooth. **A forest can approximate a smooth function
that no individual tree in it can** - which is a nice illustration that an ensemble is not merely a more
reliable version of its members.

## Failure lab: the thing trees cannot do

Every model in this module so far can be asked about an input outside the range it was trained on, and
will give an answer of some kind. A tree gives an answer too, and it is always the same answer.

### Predict before running

A model is trained on `x` between 0 and 10, where the truth is `y = 3 + 2x`. What will a tree predict at
`x = 25`?

In [ ]:
# SYNTHETIC: a perfectly straight line. TRUTH: y = 3 + 2x, noise sd 1.0
line_rng = np.random.default_rng(1)
line_x = line_rng.uniform(0, 10, 300)
line_y = 3 + 2.0 * line_x + line_rng.normal(0, 1.0, 300)

models = {
    "linear": LinearRegression().fit(line_x.reshape(-1, 1), line_y),
    "tree, depth 6": DecisionTreeRegressor(max_depth=6, random_state=0).fit(
        line_x.reshape(-1, 1), line_y),
    "forest of 200": RandomForestRegressor(n_estimators=200, random_state=0).fit(
        line_x.reshape(-1, 1), line_y),
}

rows = []
for query in [5.0, 10.0, 15.0, 25.0]:
    row = {"x": query, "truth": 3 + 2 * query}
    for label, model in models.items():
        row[label] = float(model.predict([[query]])[0])
    rows.append(row)
print("trained on x between 0 and 10\n")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: "%.3f" % v))

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.8))
wide_grid = np.linspace(0, 26, 500)

ax.scatter(line_x, line_y, s=14, alpha=0.35, color="#999999", label="the training rows")
ax.plot(wide_grid, 3 + 2 * wide_grid, color="#000000", linewidth=2.2, linestyle="--",
        label="the truth")
for label, colour in [("linear", "#009E73"), ("tree, depth 6", "#D55E00"),
                      ("forest of 200", "#0072B2")]:
    ax.plot(wide_grid, models[label].predict(wide_grid.reshape(-1, 1)), color=colour,
            linewidth=2.3, label=label)
ax.axvspan(0, 10, color="#DDEBF7", alpha=0.6, zorder=0)
ax.text(5.0, 40, "trained here", fontsize=11, ha="center", color="#0072B2",
        fontweight="bold")
ax.annotate("both trees are flat\nfrom here on", xy=(18, 23.5), xytext=(14, 36),
            fontsize=10, color="#D55E00", fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="#D55E00", linewidth=1.6))
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Past the edge of the training data, a tree is a flat line", fontsize=11.5)
ax.legend(fontsize=9, loc="upper left")
plt.tight_layout()
plt.show()

> **At `x = 25` the truth is 53. The linear model says 53.219. The tree says 23.491 and the forest says
> 23.659 - both wrong by nearly 30, and both give the same answer for every `x` above 10.**

**A tree cannot extrapolate. At all. Ever.** Its prediction is the mean of some leaf, every leaf was built
from training rows, and beyond the training range every input falls into the same outermost leaf. The
prediction is a constant, and it will remain that constant at `x = 25`, `x = 250` or `x = 25,000`.

**And the forest does not help.** Averaging 200 trees that all flatline gives a flatline. This is not a
variance problem, so the tool for variance problems does nothing.

**Why this matters more than it sounds.**

- **Time is a feature that always moves outside its training range.** A model with a date, a `days_since`
  or a rising counter in it will be extrapolating in production from the day it ships, and a tree will
  silently return whatever it learned about the last period it saw. This is the most common way a tree
  model degrades quietly.
- **Prices, volumes and counts drift.** Inflation alone puts next year's amounts outside this year's
  range.
- **It is invisible in cross-validation.** Every fold is drawn from the same range, so nothing in 04-03's
  procedure can detect it. 04-04's chronological split can.

**What to do about it:** remove or transform the trending feature - use `day of week` rather than `date`,
or a ratio rather than a level; model the trend separately and let the tree fit the residual; or use a
model that extrapolates when extrapolation is the point. **The one thing not to do is assume the model
will notice.**

## Feature importance, and how to be misled by it

A fitted forest offers `feature_importances_`, which is the total reduction in squared error each feature
achieved across all splits, weighted by how many rows passed through. It is free, it is a single tidy
table, and it is the most-misread output in applied machine learning.

In [ ]:
from sklearn.inspection import permutation_importance

# SYNTHETIC: 800 rows. TRUTH: y = 3 x useful + 2 x binary_real, noise sd 1.0.
# 'duplicate' is a near-copy of 'useful' and contributes nothing new.
# 'noise_continuous' is pure noise with a unique value per row.
trap_rng = np.random.default_rng(31)
n_trap = 800
useful = trap_rng.normal(size=n_trap)
trap = pd.DataFrame({
    "useful": useful,
    "duplicate": useful + trap_rng.normal(0, 0.05, n_trap),
    "binary_real": trap_rng.integers(0, 2, n_trap).astype(float),
    "noise_continuous": trap_rng.uniform(size=n_trap),
    "noise_binary": trap_rng.integers(0, 2, n_trap).astype(float)})
trap_y = 3.0 * trap.useful + 2.0 * trap.binary_real + trap_rng.normal(0, 1.0, n_trap)

trap_train, trap_test, trap_train_y, trap_test_y = train_test_split(
    trap, trap_y, test_size=0.3, random_state=0)
trap_forest = RandomForestRegressor(n_estimators=400, random_state=0).fit(
    trap_train, trap_train_y)

permutation = permutation_importance(trap_forest, trap_test, trap_test_y, n_repeats=20,
                                     random_state=0,
                                     scoring="neg_root_mean_squared_error")
importance = pd.DataFrame({
    "feature": trap.columns,
    "true coefficient": [3.0, 0.0, 2.0, 0.0, 0.0],
    "impurity importance": trap_forest.feature_importances_,
    "permutation importance": permutation.importances_mean})
print(importance.sort_values("impurity importance", ascending=False)
      .to_string(index=False, float_format=lambda v: "%.4f" % v))

**Two failures in one table, and only one of them is fixed by the usual advice.**

**Failure 1: correlated columns split the credit.** `useful` scores **0.4763** and its near-copy
`duplicate` scores **0.4290**, despite contributing nothing the first column does not already carry. The
tree picks whichever is convenient at each split, so the importance of a real effect is divided among its
proxies.

**Failure 2: the impurity measure is biased towards features with many possible split points.**
`binary_real` has a true coefficient of 2.0 and scores **0.0615**, while `noise_continuous` - pure
noise - scores **0.0294**. **A genuinely important binary feature scored barely twice a column with no
signal at all**, because a continuous column with 800 distinct values offers 799 chances to fit noise and
a binary column offers one.

In [ ]:
# How much of 'useful' was taken by its copy? Remove the copy and refit.
without_copy = RandomForestRegressor(n_estimators=400, random_state=0).fit(
    trap_train.drop(columns=["duplicate"]), trap_train_y)
print("importance of 'useful' with the duplicate present : %.4f"
      % trap_forest.feature_importances_[0])
print("importance of 'useful' with the duplicate removed : %.4f"
      % without_copy.feature_importances_[0])
print("\ntest R-squared with the duplicate    : %.4f"
      % trap_forest.score(trap_test, trap_test_y))
print("test R-squared without it            : %.4f"
      % without_copy.score(trap_test.drop(columns=["duplicate"]), trap_test_y))

**Removing the redundant column nearly doubles `useful`'s importance, from 0.4763 to 0.8926 - and the
model predicts essentially the same, 0.8674 against 0.8717.**

The importance moved by 87% while the model did not change at all. **Importance is a property of the
fitted tree, not of the world.**

**What permutation importance fixes, and what it does not.**

Permutation importance shuffles one column in the *held-out* rows and measures how much the error grows,
so it asks a question about predictions rather than about splits. In the table above it **fixes failure
2**: `binary_real` scores **0.5906**, clearly third and far above the two noise columns, which correctly
land at approximately zero (-0.0156 and -0.0017).

**It does not fix failure 1.** `duplicate` still scores 1.1425, because permuting it alone genuinely does
damage a model that was using it - the model cannot know to fall back on `useful`.

**So the practical rules are:**

1. **Prefer permutation importance**, computed on held-out rows, and treat impurity importance as a
   diagnostic of the fit rather than a statement about the data.
2. **Handle correlated groups before you interpret anything.** Cluster correlated features and permute a
   whole group at once, or drop redundant columns first.
3. **Never call it an effect size.** Importance has no sign, no units, and no counterfactual meaning: it
   does not say what happens if the feature increases by one, or whether the relationship is causal.

## The whole chapter on one page

In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 5.6))
ax.set_xlim(0, 10.6)
ax.set_ylim(0, 6.6)
ax.axis("off")

ax.text(5.3, 6.3, "TREES AND FORESTS", fontsize=13, fontweight="bold", ha="center")

ax.add_patch(plt.Rectangle((0.2, 3.4), 5.0, 2.6, facecolor="#D9EAD3",
                           edgecolor="#009E73", linewidth=1.6))
ax.text(2.7, 5.65, "FREE, WITH NO EFFORT", fontsize=11.5, fontweight="bold", ha="center")
for index, line in enumerate([
        "non-linearity - no terms to add",
        "interactions - splits inside splits",
        "no scaling, ever - only comparisons",
        "invariant to any monotone transform",
        "mixed feature types, side by side",
        "missing values, in some libraries"]):
    ax.text(0.35, 5.25 - index * 0.33, "+ " + line, fontsize=9.5)

ax.add_patch(plt.Rectangle((5.4, 3.4), 5.0, 2.6, facecolor="#F4CCCC",
                           edgecolor="#D55E00", linewidth=1.6))
ax.text(7.9, 5.65, "WHAT IT COSTS", fontsize=11.5, fontweight="bold", ha="center")
for index, line in enumerate([
        "CANNOT extrapolate - flat past the edge",
        "smooth trends become staircases",
        "memorises unless you stop it",
        "  (200 leaves for 200 rows)",
        "importance is not an effect size",
        "no coefficients to report at all"]):
    ax.text(5.55, 5.25 - index * 0.33, "- " + line, fontsize=9.5)

ax.text(0.2, 2.85, "THE DIALS", fontsize=11, fontweight="bold")
for index, (name, note) in enumerate([
        ("max_depth / min_samples_leaf", "capacity. Best here was depth 3, 8 leaves"),
        ("n_estimators", "more is never worse, only slower. 100 halved the variance"),
        ("max_features", "decorrelates the trees. The second idea in a forest")]):
    ax.text(0.35, 2.45 - index * 0.4, name, fontsize=9.8, fontweight="bold",
            family="monospace")
    ax.text(4.3, 2.45 - index * 0.4, note, fontsize=9.3, color="#444444")

ax.text(0.2, 1.05, "THE ONE-LINE SUMMARY", fontsize=11, fontweight="bold")
ax.text(0.2, 0.62, "A single tree is low bias and enormous variance. A forest is the same tree,",
        fontsize=10)
ax.text(0.2, 0.22, "averaged until the variance goes away - which is 05-08, as an algorithm.",
        fontsize=10)

plt.tight_layout()
plt.show()

## Common misconceptions

**"Trees need their features scaled."**
They do not, and the reason is structural rather than incidental: a tree only ever asks "is this feature
below that value?", and the answer survives any monotone transform. Scaling a tree's inputs is harmless
and pointless.

**"A random forest cannot overfit."**
More trees cannot overfit - that is genuinely true, and `n_estimators` only costs time. But a forest of
deep trees on few rows overfits like anything else, and the defaults grow trees to full depth.

**"Feature importance tells you which variables matter."**
It tells you which variables *this fitted forest used*. Removing a redundant column nearly doubled
`useful`'s importance while the predictions did not change, and a pure-noise continuous column outscored
half a real binary one.

**"The forest will handle the trend for me."**
It will predict a constant beyond the training range. This is the one failure in this chapter with no
workaround inside the model class.

**"A deeper tree is a better tree."**
Depth 3 scored 1.5867 and the unrestricted tree 2.0709, from a perfect 0.0000 on its training rows.

**"Trees are interpretable."**
A depth-3 tree is; you can read it and act on it. A depth-20 tree has up to a million leaves, and a
forest of 300 of them is not interpretable in any useful sense - which is why importance and partial
dependence tools exist, and why they need to be read with the care above.

**"Bagging gives you the 1/k variance reduction from the theory."**
It gave a factor of 2.1 here against 100 trees, because bootstrap samples overlap heavily and the trees
are correlated. The direction is right and the magnitude is not.

## Exercises

Solutions: `solutions/05_regression/05-10_trees_solutions.ipynb`.

### Quick understanding

**E1.** What does a regression tree predict in a leaf, and what criterion chooses each split?

**E2.** Give two things a tree gets for free that a linear model needs supplied by hand.

**E3.** Why does averaging trees reduce variance but not bias?

### Hand calculation

**E4.** Rows `x = [1, 2, 5, 6]`, `y = [10, 12, 20, 22]`. Compute the SSE for every candidate split and
give the winner.

**E5.** For the same data, give the depth-1 tree's prediction at `x = 3` and at `x = 100`. Explain the
second.

**E6.** A leaf holds five rows with targets 4, 6, 6, 8, 11. Give the prediction, the SSE, and the SSE if
the leaf were split into `{4, 6, 6}` and `{8, 11}`.

**E7.** A tree is fitted to 500 rows with `max_depth=None` and no other limit. How many leaves might it
have, and what will its training RMSE be?

### Coding

**E8.** Write `best_split(x, y)` returning the threshold and the SSE reduction, and check it against
`DecisionTreeRegressor(max_depth=1)` on three different datasets.

**E9.** Sweep `min_samples_leaf` from 1 to 50 on this chapter's curve data and plot train and held-out
RMSE. Compare the best result with the best `max_depth`.

**E10.** Fit a forest with `n_estimators` from 1 to 300 and plot held-out RMSE. Where does it stop
improving, and what does that tell you about how to set it?

**E11.** Sweep `max_features` on a dataset with 20 columns and report both the held-out score and the
average correlation between the trees' predictions. Explain the relationship.

**E12.** Take the shop data, add a `day_number` column that counts upward, train a forest, and evaluate
it on days after the training period. Report what happens and why.

### Interpretation

**E13.** Your forest scores far better than your linear model on the same features. Give three possible
reasons and how you would tell them apart.

**E14.** A forest's top feature by importance is `customer_id`. Say what has happened and what you would
do.

### Debugging

**E15.** A tree model performs well in backtesting and degrades steadily in production over six months.
Name the most likely cause and the check.

**E16.** Your random forest's training R-squared is 0.98 and its held-out R-squared is 0.55. Give three
things to try, in the order you would try them.

### Exam and interview reasoning

**E17.** "How does a decision tree decide where to split?" Answer in under a minute, then handle: "and
why does a random forest use a random subset of features at each split?"

### Transfer to a different situation

**E18.** You are predicting energy demand from weather, calendar and a rising installed-capacity figure.
Say what you would do with each feature type and why a forest alone is not enough.

### Explain it to someone non-technical

**E19.** In under 90 words, explain a decision tree and then a random forest.

### Optional challenge

**E20.** Implement a depth-2 regression tree from scratch - recursive best split, means in the leaves -
and verify it matches `DecisionTreeRegressor(max_depth=2)` on random data to machine precision.

**E21.** Measure how the variance reduction from bagging depends on the correlation between the trees, by
sweeping `max_features` and computing both quantities. Compare with the theoretical form
`rho * sigma^2 + (1 - rho) * sigma^2 / k`.

## Mastery check

- [ ] Find the best split in a small dataset by hand
- [ ] Explain why trees need no scaling, from the algorithm rather than by assertion
- [ ] Set a stopping rule and justify it from a depth or leaf-size sweep
- [ ] Say what bagging does to each term of the decomposition, with numbers
- [ ] Recognise when extrapolation is required and rule trees out on that basis
- [ ] Read an importance table and name its two failure modes

## What should now feel instinctive

- Reaching for a forest as the strong baseline that needs almost no preparation
- Setting a depth or leaf-size limit before fitting, not after
- Asking "will this model ever see inputs outside this range?" before shipping a tree
- Distrusting any importance number attached to correlated or high-cardinality columns
- Comparing a forest against a linear model to learn what shape the truth is

## Flashcards

| Front | Back |
|---|---|
| What a leaf predicts | the mean of the training targets in it |
| How a split is chosen | try every threshold, keep the one minimising within-group SSE |
| The eight-row example | split at 5.0, SSE 140.875 to 4.750 |
| Why no scaling | the algorithm only compares; any monotone transform gives the same tree |
| Interactions | splits inside splits. RMSE 27.94 with no interaction column supplied |
| Unrestricted tree on 200 rows | 200 leaves, training RMSE 0.0000, held-out 2.0709 |
| Bagging | bias² 0.0189 to 0.0097, variance 2.2780 to 1.0624 |
| Why not a factor of 100 | bootstrap samples overlap, so the trees are correlated |
| `max_features` | decorrelates the trees, which is what makes averaging pay |
| Extrapolation | at x = 25 the truth is 53; the tree and the forest both say ~23.5 |
| Impurity importance | splits credit between correlated columns; favours many-valued features |
| Permutation importance | fixes the cardinality bias, not the correlation one |

## Next

**05-11 · Gradient boosting.** A forest averages many trees fitted independently to the same problem.
Boosting fits them **in sequence**, each one to the errors the previous ones left - so the trees
cooperate rather than vote, and the ensemble corrects its own mistakes.

That change turns the same weak building block into the model that wins most competitions on tabular
data, and it introduces a genuinely new failure mode: a forest cannot overfit by adding trees, and a
boosted model absolutely can.